# 第 4 章 练习题答案

> 精选 2 道核心练习，巩固 GPT 结构理解。

## 练习 4.1：参数量计算

**题目**：手动计算 GPT-2 124M 各组件的参数量，解释为什么我们的实现是 163M 而官方是 124M。

In [ ]:
from src.gpt import GPTModel, GPT_CONFIG_124M
import torch

cfg = GPT_CONFIG_124M
torch.manual_seed(123)
model = GPTModel(cfg)

# 按组件统计参数量
emb_dim, ctx_len, vocab = cfg["emb_dim"], cfg["context_length"], cfg["vocab_size"]
n_layers, n_heads = cfg["n_layers"], cfg["n_heads"]
head_dim = emb_dim // n_heads

components = {
    "token embedding": vocab * emb_dim,
    "position embedding": ctx_len * emb_dim,
    "attention (qkv+proj)": n_layers * (4 * emb_dim * emb_dim + emb_dim * emb_dim),
    "FFN (2 层 linear)": n_layers * (emb_dim * 4*emb_dim + 4*emb_dim * emb_dim),
    "LayerNorm (每层2个+final)": (2*n_layers + 1) * 2 * emb_dim,
    "output head": vocab * emb_dim,   # ← 这就是和 embedding 重复的部分
}
print(f"{'组件':<28} {'参数量':>12}")
print("-" * 42)
total = 0
for name, p in components.items():
    print(f"{name:<28} {p:>12,}")
    total += p
print("-" * 42)
print(f"{'合计':<28} {total:>12,}")
print(f"\n实际 model.parameters(): {sum(p.numel() for p in model.parameters()):,}")
print(f"\n💡 output head ({vocab*emb_dim:,}) 与 token embedding 维度相同。")
print(f"   若 weight tying（共享这两者），省掉 {vocab*emb_dim:,} 参数 → 163M - 39M ≈ 124M！")

## 练习 4.2：dropout 与 train/eval 模式

**题目**：对比 train 和 eval 模式下，模型输出是否相同？为什么？

In [ ]:
import torch
from src.gpt import GPTModel, GPT_CONFIG_124M

cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim": 64, "n_layers": 1, "n_heads": 4, "context_length": 8, "drop_rate": 0.5})
torch.manual_seed(123)
model = GPTModel(cfg)
x = torch.tensor([[1, 2, 3, 4]])

# train 模式：dropout 生效，每次输出不同
model.train()
with torch.no_grad():
    out1 = model(x)
    out2 = model(x)
print(f"train 模式两次前向是否相同: {torch.allclose(out1, out2)}（dropout 随机）")

# eval 模式：dropout 关闭，输出确定
model.eval()
with torch.no_grad():
    out3 = model(x)
    out4 = model(x)
print(f"eval 模式两次前向是否相同: {torch.allclose(out3, out4)}（dropout 关闭）")
print("\n💡 eval 模式下 dropout 自动关闭 → 推理时输出确定可复现。")
print("   训练时 dropout 正则化防过拟合，推理时必须 eval() 关掉它。")